# MF Analytics Platform — Performance & Risk Metrics
## Bluestock Fintech Capstone · Metrics Notebook
Computes: Sharpe, Sortino, Alpha, Beta, VaR 95%, Max Drawdown, Rolling CAGR

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..') / 'metrics'))
from risk_metrics import (
    compute_sharpe, compute_sortino, compute_max_drawdown,
    compute_var_95, compute_beta, compute_alpha, compute_cagr,
    NIFTY_DAILY_RETURNS
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DB_PATH = Path('..') / 'mf_analytics.db'
conn = sqlite3.connect(DB_PATH)
print('Connected.')

In [ ]:
# Load all NAV data
df = pd.read_sql('''
    SELECT f.fund_id, f.scheme_name, f.amc_name, f.category, f.sub_category,
           n.nav_date, n.nav_value, n.day_change_pct
    FROM fact_nav n JOIN dim_fund f ON n.fund_id = f.fund_id
    ORDER BY f.fund_id, n.nav_date
''', conn)
df['nav_date'] = pd.to_datetime(df['nav_date'])
df['day_change_pct'] = pd.to_numeric(df['day_change_pct'], errors='coerce') / 100
print(f'Loaded {len(df):,} NAV rows for {df["fund_id"].nunique()} funds')

In [ ]:
# Compute all metrics
records = []
for fid, grp in df.groupby('fund_id'):
    grp = grp.sort_values('nav_date').dropna(subset=['day_change_pct'])
    if len(grp) < 252:
        continue
    returns = grp['day_change_pct']
    nav = grp['nav_value']
    records.append({
        'fund_id': fid,
        'scheme_name': grp['scheme_name'].iloc[0],
        'amc_name': grp['amc_name'].iloc[0],
        'category': grp['category'].iloc[0],
        'sub_category': grp['sub_category'].iloc[0],
        'sharpe_ratio': compute_sharpe(returns),
        'sortino_ratio': compute_sortino(returns),
        'max_drawdown_pct': compute_max_drawdown(nav),
        'var_95_pct': compute_var_95(returns),
        'beta': compute_beta(returns, NIFTY_DAILY_RETURNS),
        'alpha_pct': compute_alpha(returns, NIFTY_DAILY_RETURNS,
                                    compute_beta(returns, NIFTY_DAILY_RETURNS)),
        'cagr_1y_pct': compute_cagr(nav, 1),
        'cagr_3y_pct': compute_cagr(nav, 3),
        'cagr_5y_pct': compute_cagr(nav, 5),
    })

metrics = pd.DataFrame(records)
print(f'Computed metrics for {len(metrics)} funds')
metrics.head()

## Sharpe vs Sortino Scatter (Dashboard Page 2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
cats = metrics['sub_category'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(cats)))
color_map = dict(zip(cats, colors))

for cat, grp in metrics.groupby('sub_category'):
    ax.scatter(grp['sharpe_ratio'], grp['sortino_ratio'],
               c=[color_map[cat]], label=cat, s=80, alpha=0.8, edgecolors='white')

ax.axline((0,0), slope=1, color='gray', linestyle='--', alpha=0.5, label='Sharpe = Sortino')
ax.set_xlabel('Sharpe Ratio')
ax.set_ylabel('Sortino Ratio')
ax.set_title('Sharpe vs Sortino Ratio by Fund Sub-Category')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('../data/processed/chart_m1_sharpe_sortino.png', dpi=150, bbox_inches='tight')
plt.show()

## Rolling CAGR Comparison

In [ ]:
cat_cagr = metrics.groupby('sub_category')[['cagr_1y_pct','cagr_3y_pct','cagr_5y_pct']].mean().dropna()
cat_cagr = cat_cagr.sort_values('cagr_3y_pct', ascending=False)

x = np.arange(len(cat_cagr))
w = 0.25
fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - w, cat_cagr['cagr_1y_pct'], w, label='1Y CAGR', color='#4575b4')
ax.bar(x,     cat_cagr['cagr_3y_pct'], w, label='3Y CAGR', color='#f46d43')
ax.bar(x + w, cat_cagr['cagr_5y_pct'], w, label='5Y CAGR', color='#1a9850')
ax.set_xticks(x)
ax.set_xticklabels(cat_cagr.index, rotation=30, ha='right')
ax.set_ylabel('CAGR (%)')
ax.set_title('Rolling CAGR by Sub-Category — 1Y / 3Y / 5Y')
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../data/processed/chart_m2_cagr.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey Insight: Mid-cap 3Y CAGR vs Large-cap:')
mid = cat_cagr.loc['Mid Cap','cagr_3y_pct'] if 'Mid Cap' in cat_cagr.index else float('nan')
large = cat_cagr.loc['Large Cap','cagr_3y_pct'] if 'Large Cap' in cat_cagr.index else float('nan')
print(f'  Mid Cap: {mid:.2f}%  |  Large Cap: {large:.2f}%  |  Alpha: {mid-large:.2f}%')

## VaR & Max Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

risk_cat = metrics.groupby('sub_category')[['var_95_pct','max_drawdown_pct']].mean().sort_values('var_95_pct')

risk_cat['var_95_pct'].plot(kind='barh', ax=axes[0], color='#d73027')
axes[0].set_title('VaR 95% by Sub-Category (lower = safer)')
axes[0].set_xlabel('VaR 95% (%)')

risk_cat['max_drawdown_pct'].sort_values().plot(kind='barh', ax=axes[1], color='#fc8d59')
axes[1].set_title('Max Drawdown % by Sub-Category')
axes[1].set_xlabel('Max Drawdown (%)')

plt.tight_layout()
plt.savefig('../data/processed/chart_m3_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## Benchmark Comparison (O7)

In [ ]:
bench = pd.read_sql('''
    SELECT f.scheme_name, f.benchmark_index, f.sub_category,
           AVG(n.day_change_pct) * 252 as annualised_return
    FROM fact_nav n JOIN dim_fund f ON n.fund_id = f.fund_id
    WHERE n.day_change_pct IS NOT NULL
    GROUP BY f.fund_id ORDER BY f.benchmark_index
''', conn)

# Synthetic benchmark returns
benchmark_returns = {
    'Nifty 50': 13.5,
    'Nifty 100': 13.0,
    'Nifty Midcap 150': 16.2,
    'BSE SmallCap': 18.0,
    'Nifty 50 Hybrid': 11.5,
    'Nifty LargeMidcap 250': 14.8,
    'Nifty 500': 14.2,
}

bench['benchmark_return'] = bench['benchmark_index'].map(benchmark_returns)
bench['alpha'] = bench['annualised_return'] - bench['benchmark_return']

fig, ax = plt.subplots(figsize=(12, 5))
bench_grp = bench.groupby('benchmark_index')[['annualised_return','benchmark_return']].mean()
bench_grp['alpha'] = bench_grp['annualised_return'] - bench_grp['benchmark_return']
x = np.arange(len(bench_grp))
w = 0.35
ax.bar(x - w/2, bench_grp['annualised_return'], w, label='Active Fund (annualised)', color='steelblue')
ax.bar(x + w/2, bench_grp['benchmark_return'], w, label='Benchmark Return', color='coral', alpha=0.8)
for i, (idx, row) in enumerate(bench_grp.iterrows()):
    ax.text(i, max(row['annualised_return'], row['benchmark_return']) + 0.3,
            f'α={row["alpha"]:+.1f}%', ha='center', fontsize=9, fontweight='bold', color='green')
ax.set_xticks(x)
ax.set_xticklabels(bench_grp.index, rotation=20, ha='right')
ax.set_ylabel('Annualised Return (%)')
ax.set_title('Fund Returns vs Benchmark — Alpha by Index (O7)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/chart_m4_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save outputs
metrics.sort_values('sharpe_ratio', ascending=False).to_csv('../metrics/fund_sharpe_ranks.csv', index=False)
metrics[['scheme_name','amc_name','category','sub_category','var_95_pct','max_drawdown_pct','beta']]\
    .sort_values('var_95_pct').to_csv('../metrics/var_drawdown_summary.csv', index=False)
print('Saved fund_sharpe_ranks.csv and var_drawdown_summary.csv')
conn.close()